# Parameter Golf — Setup & Pull from GitHub

Run this notebook first to set up the environment on Colab.
It clones both the official repo (for data/baseline) and our aux losses repo.

**Usage:**
1. Upload this notebook to Google Drive
2. Open in Colab (GPU runtime)
3. Run all cells
4. Then open `parameter_golf_experiments.ipynb` from the cloned repo

## 1. Mount Google Drive (persistent storage for logs/results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Persistent storage on Drive for logs and results
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/results', exist_ok=True)
print(f'Drive directory: {DRIVE_DIR}')

## 2. Clone Official Parameter Golf Repo (data + baseline)

In [ ]:
%cd /content

# Clone official repo if not already present
if not os.path.exists('/content/parameter-golf'):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
else:
    %cd /content/parameter-golf
    !git pull
    %cd /content

print('Official repo ready')

## 3. Clone Our Auxiliary Losses Repo

In [ ]:
%cd /content

# Clone our aux losses repo if not already present
if not os.path.exists('/content/parameter-golf-aux'):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git
else:
    %cd /content/parameter-golf-aux
    !git pull
    %cd /content

print('Aux losses repo ready')

## 4. Copy Our Files into the Official Repo

In [ ]:
import shutil

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

# Copy our training scripts
shutil.copy2(f'{AUX}/train_gpt_aux.py', f'{OFFICIAL}/train_gpt_aux.py')
shutil.copy2(f'{AUX}/train_gpt_sota.py', f'{OFFICIAL}/train_gpt_sota.py')

# Copy aux_losses module
if os.path.exists(f'{OFFICIAL}/aux_losses'):
    shutil.rmtree(f'{OFFICIAL}/aux_losses')
shutil.copytree(f'{AUX}/aux_losses', f'{OFFICIAL}/aux_losses')

# Copy scripts (experiment runner, data analysis)
for src_file in ['scripts/experiment_runner.py', 'scripts/analyze_training_data.py']:
    src = f'{AUX}/{src_file}'
    if os.path.exists(src):
        os.makedirs(f'{OFFICIAL}/{os.path.dirname(src_file)}', exist_ok=True)
        shutil.copy2(src, f'{OFFICIAL}/{src_file}')

# Copy experiment configs
for cfg in ['sweep_config.json', 'sweep_config_a100.json', 'sweep_config_t4.json']:
    src = f'{AUX}/experiments/{cfg}'
    if os.path.exists(src):
        os.makedirs(f'{OFFICIAL}/experiments', exist_ok=True)
        shutil.copy2(src, f'{OFFICIAL}/experiments/{cfg}')

# Symlink logs to Drive for persistence
logs_dir = f'{OFFICIAL}/logs'
if os.path.exists(logs_dir) and not os.path.islink(logs_dir):
    # Move any existing logs to Drive first
    for f in os.listdir(logs_dir):
        shutil.copy2(f'{logs_dir}/{f}', f'{DRIVE_DIR}/logs/{f}')
    shutil.rmtree(logs_dir)
if not os.path.exists(logs_dir):
    os.symlink(f'{DRIVE_DIR}/logs', logs_dir)

print('Files copied into official repo')
print(f'Logs will persist at: {DRIVE_DIR}/logs/')

## 5. Install Dependencies & Download Data

In [ ]:
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

# Install FlashAttention 2 for A100 (takes ~5 min to compile, but huge speedup)
import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
if "A100" in gpu_name or "H100" in gpu_name or "A10" in gpu_name:
    print(f"Installing flash-attn for {gpu_name}...")
    !pip install -q flash-attn --no-build-isolation
else:
    print(f"GPU: {gpu_name} — skipping flash-attn (will use PyTorch SDPA fallback)")

In [ ]:
%cd /content/parameter-golf

# Download 1 shard for quick experiments (use --train-shards 10 for fuller runs)
SHARDS = 1
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards {SHARDS}

## 6. Verify Everything Works

In [ ]:
%cd /content/parameter-golf

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print()

# Syntax check
!python3 -c "import py_compile; py_compile.compile('train_gpt_aux.py', doraise=True); print('train_gpt_aux.py: OK')"
!python3 -c "from aux_losses import focal_cross_entropy, inter_layer_decorrelation_loss, representation_rank_loss, unigram_kl_loss; print('aux_losses imports: OK')"

# Check data
import glob
train_files = glob.glob('data/datasets/fineweb10B_sp1024/fineweb_train_*.bin')
val_files = glob.glob('data/datasets/fineweb10B_sp1024/fineweb_val_*.bin')
print(f'Train shards: {len(train_files)}')
print(f'Val shards: {len(val_files)}')
print()
print('Setup complete. Ready to run experiments.')

## 7. Quick Pull (run this cell to update code without re-running everything)

After pushing changes from WSL, run just this cell to pull the latest code.

In [ ]:
import shutil, os

# Pull latest from our repo
%cd /content/parameter-golf-aux
!git pull

# Re-copy into official repo
OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'

shutil.copy2(f'{AUX}/train_gpt_aux.py', f'{OFFICIAL}/train_gpt_aux.py')
shutil.copy2(f'{AUX}/train_gpt_sota.py', f'{OFFICIAL}/train_gpt_sota.py')
if os.path.exists(f'{OFFICIAL}/aux_losses'):
    shutil.rmtree(f'{OFFICIAL}/aux_losses')
shutil.copytree(f'{AUX}/aux_losses', f'{OFFICIAL}/aux_losses')

# Re-copy scripts and configs
for src_file in ['scripts/experiment_runner.py', 'scripts/analyze_training_data.py']:
    src = f'{AUX}/{src_file}'
    if os.path.exists(src):
        os.makedirs(f'{OFFICIAL}/{os.path.dirname(src_file)}', exist_ok=True)
        shutil.copy2(src, f'{OFFICIAL}/{src_file}')
for cfg in ['sweep_config.json', 'sweep_config_a100.json', 'sweep_config_t4.json']:
    src = f'{AUX}/experiments/{cfg}'
    if os.path.exists(src):
        shutil.copy2(src, f'{OFFICIAL}/experiments/{cfg}')

%cd /content/parameter-golf
!python3 -c "import py_compile; py_compile.compile('train_gpt_aux.py', doraise=True); py_compile.compile('train_gpt_sota.py', doraise=True); print('Updated & verified OK')"

---

## Experiments (via Experiment Runner)

The runner automatically selects the right config for your GPU.
It skips already-completed runs, so you can re-run safely.

In [ ]:
%cd /content/parameter-golf
import torch

# Auto-detect GPU and pick config
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

if vram_gb >= 40:
    CONFIG = "experiments/sweep_config_a100.json"
    print(f"GPU: {gpu_name} ({vram_gb:.0f}GB) → Using A100 config (786K batch tokens)")
elif vram_gb >= 14:
    CONFIG = "experiments/sweep_config_t4.json"
    print(f"GPU: {gpu_name} ({vram_gb:.0f}GB) → Using T4 config (131K batch tokens)")
else:
    CONFIG = "experiments/sweep_config_t4.json"
    print(f"GPU: {gpu_name} ({vram_gb:.0f}GB) → Using T4 config (conservative)")

### Step 1: Run Baselines (both SOTA and our script)

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} --filter "baseline" --seeds 42,1337,7

### Step 2: Screen all losses (1 seed each)

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} --filter "focal|decorr|rank|unigram" --seeds 42

### Step 3: Confirm winners with 3 seeds (edit filter to match winners from Step 2)

In [ ]:
# Edit the filter to match winning experiments from Step 2
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --config {CONFIG} --filter "focal_g1|decorr_001" --seeds 42,1337,7

### Step 4: Generate full report

In [ ]:
%cd /content/parameter-golf
!python3 scripts/experiment_runner.py --report-only

### Save results to Drive

In [ ]:
import shutil
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

# Copy auto-generated report and JSON to Drive
for f in ['experiments/results_auto.md', 'experiments/results_auto.json']:
    if os.path.exists(f):
        shutil.copy2(f, f'{DRIVE_DIR}/results/{os.path.basename(f)}')
        print(f'Saved {f} → Drive')

# Display the report
if os.path.exists('experiments/results_auto.md'):
    from IPython.display import Markdown, display
    display(Markdown(open('experiments/results_auto.md').read()))